# Where does GPT-2 break down for its size?

**To run:** tap **Runtime → Run all** (on a phone, the menu may be under ⋮ or ☰). It takes about 10–20 minutes.
Keep this tab open while it runs.

Faster (optional): **Runtime → Change runtime type → T4 GPU** before running. Without a GPU it still works,
but it uses fewer facts and skips GPT-2 XL.

When it finishes, scroll to the bottom and copy the results text back to Claude.

In [ ]:
# 1. Get the code
!rm -rf safeinterp && git clone -q -b claude/gifted-darwin-ithng2 https://github.com/nyancatspace/safeinterp
%cd safeinterp
!pip install -q -e . 2>&1 | tail -1

In [ ]:
# 2. Settings: picks sizes based on whether a GPU is available
import torch
GPU = torch.cuda.is_available()
MODELS = "gpt2,gpt2-medium,gpt2-large,gpt2-xl" if GPU else "gpt2,gpt2-medium,gpt2-large"
LIMIT = 5000 if GPU else 800
print("GPU:", torch.cuda.get_device_name(0) if GPU else "none (slower, fewer facts)")
print("models:", MODELS, "| facts:", LIMIT)

In [ ]:
# 3. Download CounterFact (facts from the ROME paper)
import json, os, urllib.request
if not os.path.exists("counterfact.json"):
    try:
        urllib.request.urlretrieve("https://rome.baulab.info/data/dsets/counterfact.json", "counterfact.json")
    except Exception as e:
        print("main download failed, trying Hugging Face copy:", e)
        from datasets import load_dataset
        json.dump(list(load_dataset("azhx/counterfact", split="train")), open("counterfact.json", "w"))
print(len(json.load(open("counterfact.json"))), "facts downloaded")

In [ ]:
# 4. Run the comparison (the slow step)
!python -m safeinterp breakdown --models {MODELS} --facts counterfact.json --limit {LIMIT} --out results 2>&1 | grep --line-buffered -v "Loading weights\|Warning\|warn"

In [ ]:
# 5. Plot: answer rank by layer for each size (lower = the model is closer to saying the answer)
from IPython.display import Image, display
if os.path.exists("results/lens_by_size.png"):
    display(Image("results/lens_by_size.png"))

In [ ]:
# 6. Results: copy everything below and paste it to Claude
print(open("results/report.md").read())